In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import torch

# Resnet-18

In [ ]:


print("================ DIAGNÓSTICO ================")

# 1. Verificar Entorno Virtual
ruta_python = sys.executable
print(f"📦 Ruta del Kernel (Python): \n   {ruta_python}")

if "venvvpc2" in ruta_python:
    print("   ✅ CORRECTO: Estás ejecutando el código dentro del entorno virtual.")
else:
    print("   ❌ ERROR: No estás en el entorno virtual. Revisa el Kernel seleccionado.")

print("\n---------------------------------------------")

# 2. Verificar Placa de Video con PyTorch
print(f"🔥 Versión de PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    nombre_gpu = torch.cuda.get_device_name(0)
    print("   ✅ CORRECTO: PyTorch tiene contacto con tu GPU.")
    print(f"   🎮 Tarjeta de video detectada: {nombre_gpu}")
else:
    print("   ❌ ERROR: PyTorch está usando el CPU.")
    print("      Asegúrate de haber instalado la versión con CUDA.")

print("=============================================")


In [ ]:
# Bloque 1


import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights

# 1. Función de Recorte Inteligente (Crop Biológico)
def crop_espectrograma(img):
    # crop(imagen, arriba, izquierda, altura_deseada, anchura_deseada)
    # Top=0 (conserva las frecuencias altas), Left=16 (corta 16px del arranque temporal)
    # Al pedir 224x224 a partir de ahí, recorta automáticamente las bandas inferiores y el excedente derecho.
    return F.crop(img, top=0, left=16, height=224, width=224)

# 2. Definimos nuestras transformaciones ensambladas
transformaciones = transforms.Compose([
    transforms.Lambda(crop_espectrograma),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# 3. Leemos las carpetas (ruta relativa)
ruta_train = "Dataset_Imagenes_Robustas/Training"
ruta_val = "Dataset_Imagenes_Robustas/Validation"

train_dataset = datasets.ImageFolder(root=ruta_train, transform=transformaciones)
val_dataset = datasets.ImageFolder(root=ruta_val, transform=transformaciones)

# Guardamos dinámica y automáticamente la cantidad de clases para que el Bloque B no explote
nombres_clases = train_dataset.classes
num_clases = len(nombres_clases)

print("✅ ¡Bloque A ejecutado con éxito!")
print(f"🦅 Especies detectadas ({num_clases}): {nombres_clases}")

# 4. Empaquetamos en DataLoaders (lotes de 32, mezcla aleatoria para entrenamiento)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Bloque 2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
# (Asegúrate de guardar en num_ftrs el número de neuronas de entrada como veníamos haciendo)
num_ftrs = model.fc.in_features

# REEMPLAZA EL model.fc VIEJO POR ESTO:
model.fc = nn.Sequential(
    nn.Dropout(p=0.4),  # El 40% de las conexiones se apagarán aleatoriamente en cada lote
    nn.Linear(num_ftrs, num_clases)
)

model = model.to(device)
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(model.parameters(), lr=0.0001)
num_epocas = 20
# ¡NUEVO! Listas para guardar las métricas de la historia
hist_train_loss = []
hist_val_loss = []
hist_val_acc = []
# --- BUCLE DE ENTRENAMIENTO ---
for epoca in range(num_epocas):
    print(f"Época {epoca+1}/{num_epocas} - Procesando... ⏳")
    
    # ENTRENAMIENTO
    model.train()
    perdida_train = 0.0
    total_train = 0
    
    for audios_img, etiquetas in train_loader:
        audios_img, etiquetas = audios_img.to(device), etiquetas.to(device)
        
        optimizador.zero_grad()
        salidas = model(audios_img)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()
        
        perdida_train += perdida.item() * audios_img.size(0)
        total_train += etiquetas.size(0)
        
    # VALIDACIÓN
    model.eval()
    perdida_val = 0.0
    aciertos_val = 0
    total_val = 0
    
    with torch.no_grad():
        for audios_img, etiquetas in val_loader:
            audios_img, etiquetas = audios_img.to(device), etiquetas.to(device)
            
            salidas = model(audios_img)
            perdida = criterio(salidas, etiquetas)
            
            perdida_val += perdida.item() * audios_img.size(0)
            _, predicciones = torch.max(salidas, 1)
            aciertos_val += torch.sum(predicciones == etiquetas.data)
            total_val += etiquetas.size(0)
            
    # Calculamos promedios
    epoch_t_loss = perdida_train / total_train
    epoch_v_loss = perdida_val / total_val
    epoch_v_acc = aciertos_val.double().item() / total_val
    
    # ¡NUEVO! Guardamos la historia del epoch en nuestras listas
    hist_train_loss.append(epoch_t_loss)
    hist_val_loss.append(epoch_v_loss)
    hist_val_acc.append(epoch_v_acc)
    
    print(f"✅ Loss: Train {epoch_t_loss:.4f} | Val {epoch_v_loss:.4f} || Acc: Val {epoch_v_acc:.4f}\n")
# Guardado final
torch.save(model.state_dict(), "modelo_resnet18_aves_20_LR4.pth")
print("¡Entrenamiento y Guardado finalizado! 🚀")


In [ ]:


# Nos aseguramos de que el eje X coincida con la cantidad de épocas reales que corriste
cantidad_epocas_reales = len(hist_train_loss)
epocas_x = np.arange(1, cantidad_epocas_reales + 1)

# Creamos la figura y el eje principal (para las Pérdidas / Loss)
fig, ax1 = plt.subplots(figsize=(10, 6))

color_tloss = 'tab:red' # Rojo
color_vloss = 'tab:orange' # Naranja

ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Loss (Pérdida)', color='black', fontsize=12)

# Dibujamos las curvas de Loss (Entrenamiento vs Validación)
linea1 = ax1.plot(epocas_x, hist_train_loss, color=color_tloss, marker='o', label='Train Loss', linewidth=2)
linea2 = ax1.plot(epocas_x, hist_val_loss, color=color_vloss, marker='s', label='Validation Loss', linewidth=2)
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, linestyle='--', alpha=0.6)

# Creamos un segundo eje Y "espejo" (para la Precisión / Accuracy) compartiendo el mismo eje X
ax2 = ax1.twinx()  

color_vacc = 'tab:blue' # Azul para diferenciarlo bien
ax2.set_ylabel('Accuracy (Precisión)', color=color_vacc, fontsize=12)

# Dibujamos la curva de la Precisión en el nuevo eje
linea3 = ax2.plot(epocas_x, hist_val_acc, color=color_vacc, marker='^', label='Validation Accuracy', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color_vacc)

# Fijamos el límite de la precisión de 0.0 a 1.0 (o un poquito más para que no se pegue al techo)
ax2.set_ylim([0.0, 1.05])

# Unificamos todas las etiquetas en una sola cajita de leyenda
lineas = linea1 + linea2 + linea3
titulos = [l.get_label() for l in lineas]
ax1.legend(lineas, titulos, loc='center right', framealpha=0.9)

plt.title('Rendimiento del Modelo: Train Loss vs Val Loss & Validation Accuracy', fontsize=14, pad=15)
plt.xticks(epocas_x) # Asegura que muestre números enteros en los ticks inferiores
fig.tight_layout()


plt.show()


In [ ]:
import torch
from sklearn.metrics import classification_report

print("Calculando las predicciones de Validación... ⏳")

# 1. Aseguramos que el modelo esté en "Modo Examen"
model.eval()

y_verdaderos = []
y_predichos = []

# 2. Pasamos todas las imágenes de validación por la red (sin gastar RAM extra)
with torch.no_grad():
    for audios_img, etiquetas in val_loader:
        audios_img = audios_img.to(device)
        
        # Le pedimos la predicción
        salidas = model(audios_img)
        
        # Nos quedamos con la especie que tenga máxima probabilidad
        _, predicciones = torch.max(salidas, 1)
        
        # Guardamos en nuestras listas las etiquetas posta vs las del modelo
        y_verdaderos.extend(etiquetas.cpu().numpy())
        y_predichos.extend(predicciones.cpu().numpy())

# 3. Construimos el cuadro comparativo final (Benchmark)
print("\n" + "="*50)
print("📊 REPORTE DE CLASIFICACIÓN (BENCHMARK)")
print("="*50)

# Le pasamos los verdaderos, los predichos, y la lista con los nombres de las especies
cuadro_metricas = classification_report(
    y_verdaderos, 
    y_predichos, 
    target_names=nombres_clases,
    zero_division=0 
)

print(cuadro_metricas)
print("="*50)


In [ ]:
import torch
import pandas as pd
from sklearn.metrics import classification_report

# NOMBRES DE TUS ARCHIVOS GUARDADOS (¡modifícalos si los llamaste distinto!)
ruta_modelo_20 = "modelo_resnet18_aves_20.pth"
ruta_modelo_20_LR4 = "modelo_resnet18_aves_20_LR4.pth"

# ==========================================
# FUNCIÓN AUXILIAR PARA EVALUAR UN MODELO
# ==========================================
def obtener_metricas_df(ruta_pesos, nombre_etiqueta):
    print(f"Cargando y evaluando: {nombre_etiqueta} ... ⏳")
    
    # 1. Cargamos el "cerebro" específico en nuestra red
    model.load_state_dict(torch.load(ruta_pesos))
    model.eval() # Modo examen
    
    y_verdaderos = []
    y_predichos = []
    
    # 2. Examinamos todo el lote de Validación
    with torch.no_grad():
        for audios_img, etiquetas in val_loader:
            audios_img = audios_img.to(device)
            salidas = model(audios_img)
            _, predicciones = torch.max(salidas, 1)
            
            y_verdaderos.extend(etiquetas.cpu().numpy())
            y_predichos.extend(predicciones.cpu().numpy())

    # 3. Extraemos las métricas usando Sklearn, pero en formato Diccionario
    reporte_dic = classification_report(
        y_verdaderos, 
        y_predichos, 
        target_names=nombres_clases,
        output_dict=True, # Magia para poder meterlo en Pandas
        zero_division=0
    )
    
    # 4. Lo convertimos en una tablita de Pandas
    df_reporte = pd.DataFrame(reporte_dic).transpose()
    
    # Le ponemos un "Techo" al nombre de las columnas para saber de qué modelo son
    df_reporte.columns = pd.MultiIndex.from_product([[nombre_etiqueta], df_reporte.columns])
    
    return df_reporte


# ==========================================
# EJECUCIÓN
# ==========================================
try:
    # Generamos la tabla para el Modelo 1 (10 épocas)
    df_m10 = obtener_metricas_df(ruta_modelo_20, "Modelo 10 Épocas")
    
    # Generamos la tabla para el Modelo 2 (20 épocas con Dropout)
    df_m20 = obtener_metricas_df(ruta_modelo_20_LR4, "Modelo 20 Épocas LR 10e-4")
    
    # Juntamos las dos tablas de costado (concat axis=1)
    tabla_comparativa = pd.concat([df_m10, df_m20], axis=1)
    
    print("\n✅ ¡Evaluación completada! Aquí tienes el Benchmark Comparativo:")
    
    # Redondeamos a 3 decimales para que se vea más lindo y lo pintamos
    display(tabla_comparativa.round(3))

except FileNotFoundError as e:
    print(f"\n❌ ERROR: ¡Alto ahí! Jupyter no encuentra el archivo guardado.")
    print("Revisa los nombres de las variables  y ruta_modelo")


In [ ]:
# Cuenta todos los parámetros totales (congelados y no congelados)
total_params = sum(p.numel() for p in model.parameters())

# Cuenta cuántos de esos parámetros se modificaron en tu entrenamiento (¡Fueron todos!)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🧠 Total de parámetros matemáticos de la red: {total_params:,}")
print(f"🔥 Parámetros que aprendieron de tus aves:    {trainable_params:,}")


In [ ]:
import torch
import pandas as pd
import os
from sklearn.metrics import classification_report
from torchvision import models
import torch.nn as nn

# ==========================================
# 1. PARAMETRIZACIÓN DEL ENTORNO
# ==========================================
# Rescatamos la línea perdida: conectarnos a la tarjeta gráfica
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# NOMBRE DEL ARCHIVO DE TU MODELO A EVALUAR
ruta_mejor_modelo = "modelo_resnet18_aves_20_LR4.pth"

print(f"Enviando proceso a: {device}")
print(f"Cargando el modelo desde: {ruta_mejor_modelo} ... ⏳")

try:
    # ==========================================
    # 2. CREACIÓN DEL ENVASE FÍSICO
    # ==========================================
    model = models.resnet18() 
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),  
        nn.Linear(model.fc.in_features, num_clases) 
    )
    model = model.to(device) 

    # ==========================================
    # 3. CARDAGO Y EVALUACIÓN
    # ==========================================
    model.load_state_dict(torch.load(ruta_mejor_modelo))
    model.eval() 
    
    y_verdaderos = []
    y_predichos = []
    
    with torch.no_grad():
        for audios_img, etiquetas in val_loader:
            audios_img = audios_img.to(device)
            salidas = model(audios_img)
            
            _, predicciones = torch.max(salidas, 1)
            
            y_verdaderos.extend(etiquetas.cpu().numpy())
            y_predichos.extend(predicciones.cpu().numpy())

    # ==========================================
    # 4. EXTRACCIÓN DE MÉTRICAS (SKLEARN)
    # ==========================================
    reporte_dic = classification_report(
        y_verdaderos, 
        y_predichos, 
        target_names=nombres_clases,
        output_dict=True, 
        zero_division=0
    )
    
    df_reporte = pd.DataFrame(reporte_dic).transpose()
    df_reporte = df_reporte.round(3)
    
    print("\n✅ ¡Evaluación completada! Este es tu Classification Report:\n")
    display(df_reporte)

    # ==========================================
    # 5. MAGIA COLABORATIVA: EXPORTAR A CSV
    # ==========================================
    nombre_base = os.path.splitext(os.path.basename(ruta_mejor_modelo))[0]
    nombre_csv = f"reporte_{nombre_base}.csv"
    
    df_reporte.to_csv(nombre_csv)
    print(f"\n📂 ¡Éxito! Tu tabla física en CSV fue exportada como: {nombre_csv}")

except FileNotFoundError:
    print(f"\n❌ ERROR IMPORTANTE: Jupyter no encontró el archivo físico. Verifica que '{ruta_mejor_modelo}' exista.")
except NameError as e:
    print(f"\n❌ ERROR DE RAM: {e}. Vuelve a ejecutar la celda que tiene los dataloaders primero.")
